# sindyc_07: what limits the world model, and what improves it

Follows `sindyc_06` (R and F need no memory; the alive rule is near its floor after a multi-step
refit). Question here: **where is the remaining error of `model_derived`, and which changes
remove it?** Every claimed improvement is checked on the validation battles with paired
bootstrap intervals and then on a test set not used for any choice in this notebook (the six
`cf_*` counterfactual sets: 600 battles, each branched six ways).

Method throughout: roll the model out from a start point every second of each battle, under the
real intents, and compare with what happened. Metrics: health error (mean |error|, both sides),
and the R² of the *change* in the margin (Blue − Red health), at 1 / 2 / 4 / 8 s.

Helpers are in `notebooks/sindyc_07_lab.py`: a lean simulator of `model_derived` with every
number exposed as a parameter, and fitting on rollout error.

All data is exp06 (exploratory, `allow_dirty`). Re-run after re-collection.

In [ ]:
import pickle
import time
import warnings

import numpy as np
import pandas as pd
import sindyc_07_lab as lab
from scipy.optimize import least_squares
from scipy.special import ndtr
from scipy.stats import spearmanr
from sindyc_07_lab import (
    PROCESSED,
    SIDES,
    H,
    base_params,
    current,
    metrics,
    paired_bootstrap,
    refit,
    residuals,
    scale_params,
    simulate,
    subsample,
)
from sklearn.ensemble import HistGradientBoostingRegressor as HGB

from sindy_wm.envs.intents import INTENTS
from sindy_wm.models.intent_model import (
    DT,
    REACH_COLS,
    SHAPE,
    STATE,
    add_controls,
    control_vector,
    r2,
    team_shares,
)

warnings.filterwarnings("ignore")
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 120)

B = {name: lab.load_batch(name) for name in ("train", "val", "test")}
Bv, Bt, Bfit = B["val"], B["train"], subsample(B["train"], 20000)
for name, b in B.items():
    print(f"{name}: {len(b['X0'])} start points from {len(np.unique(b['group']))} battles")

pA = base_params()
Y_ref = current.simulate(Bv["X0"], Bv["shape0"], Bv["U"], reach0=Bv["reach0"])
print(
    "lean simulator vs IntentModel.simulate, max |difference|:", np.nanmax(np.abs(simulate(pA, Bv) - Y_ref))
)


def carry(M):
    """Carry the last real value forward over the NaNs after a battle's end."""
    M = M.copy()
    for n in range(1, M.shape[1]):
        bad = np.isnan(M[:, n])
        M[:, n][bad] = M[:, n - 1][bad]
    return M


def show(out, cols=("health", "margin")):
    T = pd.DataFrame(out).T
    return T[[c for c in T.columns if any(k in c for k in cols)]].round(3)

## 1 The ceiling: how well can *any* predictor do with this state?

Gradient-boosted trees predict health and the margin h seconds ahead **directly** from what the
model starts from: the team state (health, alive, distance), formation (depth, width), measured
units in reach, and the intents played in each second of the horizon. Trained on the training
start points. This is not a model to use; it is the benchmark that says how much the state allows.

In [ ]:
def start_features(Bx, n):
    U = Bx["U"][:, :n]
    return np.column_stack(
        [Bx["X0"], Bx["shape0"], Bx["reach0"]] + [U[:, k : k + 8].mean(1) for k in range(0, n, 8)]
    )


ceiling = []
for name in ("val", "test"):
    Bx = B[name]
    mm = metrics(simulate(pA, Bx), Bx)
    for h in H:
        n = round(h / DT)
        okt, ok = ~np.isnan(Bt["Y"][:, n, 0]), ~np.isnan(Bx["Y"][:, n, 0])
        Xt, Xv = start_features(Bt, n)[okt], start_features(Bx, n)[ok]
        pred = [
            HGB(max_iter=400, random_state=0).fit(Xt, Bt["Y"][okt, n, j] - Bt["X0"][okt, j]).predict(Xv)
            + Bx["X0"][ok, j]
            for j in (0, 1)
        ]
        real = Bx["Y"][ok, n, 0] - Bx["Y"][ok, n, 1] - (Bx["X0"][ok, 0] - Bx["X0"][ok, 1])
        pm = pred[0] - pred[1] - (Bx["X0"][ok, 0] - Bx["X0"][ok, 1])
        ceiling.append(
            {
                "set": name,
                "horizon (s)": h,
                "health: model_derived": mm[f"health {h}s"],
                "health: trees": np.mean([np.abs(pred[j] - Bx["Y"][ok, n, j]).mean() for j in (0, 1)]),
                "margin R²: model_derived": mm[f"margin R2 {h}s"],
                "margin R²: trees": 1 - np.sum((pm - real) ** 2) / np.sum((real - real.mean()) ** 2),
            }
        )
ceiling = pd.DataFrame(ceiling).set_index(["set", "horizon (s)"])
ceiling.round(3)

## 2 Where is the gap? Oracles

`model_derived` unchanged, with parts replaced by the real values at every step:

- **true share**: the real share of units in reach (R / A) of both sides. Unlike "true R" in
  `sindyc_06`, it does not carry who died.
- **true distance**, **true formation** (depth, width), and both.

Then the same four input cases with the formula closure swapped for a **tree closure**: trees
that map the closure's own inputs (distance, depths, widths, alive counts) to the share. That
separates "the closure's form" from "the closure's inputs".

In [ ]:
Yt, St = carry(Bv["Y"]), carry(Bv["Yshape"])
share_true = np.divide(
    carry(Bv["Yreach"]), Yt[:, :, 2:4], out=np.zeros_like(Bv["Yreach"]), where=Yt[:, :, 2:4] > 0
)

rows_tr = add_controls(pd.read_parquet(PROCESSED / "train.parquet"))
rows_tr = rows_tr[(rows_tr["alive_blue"] > 0) & (rows_tr["alive_red"] > 0)]
rows_va = add_controls(pd.read_parquet(PROCESSED / "val.parquet"))
rows_va = rows_va[(rows_va["alive_blue"] > 0) & (rows_va["alive_red"] > 0)].copy()


def closure_inputs(X, S, side):
    Hb, Hr, Ab, Ar, d = X.T
    db, dr, wb, wr = S.T
    return (
        np.column_stack([d, db, dr, wb, wr, Ar, Ab])
        if side == "blue"
        else np.column_stack([d, dr, db, wr, wb, Ab, Ar])
    )


fit_rows = rows_tr.sample(250_000, random_state=0)
tree_closure = {
    s: HGB(max_iter=300, random_state=0).fit(
        closure_inputs(fit_rows[STATE].to_numpy(), fit_rows[SHAPE].to_numpy(), s),
        (fit_rows[f"reach_{s}"] / fit_rows[f"alive_{s}"]).to_numpy(),
    )
    for s in SIDES
}


def tree_share(X, S, u, p):
    return np.column_stack([np.clip(tree_closure[s].predict(closure_inputs(X, S, s)), 0, 1) for s in SIDES])


oracles = {"model_derived, true share": metrics(simulate(pA, Bv, truth={"share": share_true}), Bv)}
for cname, fn in (("formula closure", None), ("tree closure", tree_share)):
    lab.SHARE_FN = fn
    for iname, truth in (
        ("model distance, held formation", {}),
        ("model distance, true formation", {"shape": St}),
        ("true distance, held formation", {"distance": Yt[:, :, 4]}),
        ("true distance, true formation", {"distance": Yt[:, :, 4], "shape": St}),
    ):
        oracles[(cname, iname)] = metrics(simulate(pA, Bv, truth=truth), Bv)
lab.SHARE_FN = None
show(oracles)

## 3 The closure: is it missing inputs, or is its form wrong?

Per step, on the validation battles: R² of units in reach for the formula, and for trees given
(a) exactly the formula's inputs and (b) more (intent, direction of travel). Then the formula's
squared error split by regime: Blue **scattered** (depth > 1.5 or width > 3) or compact.

In [ ]:
from sindy_wm.models.intent_model import add_rates  # noqa: E402

tr_d = add_rates(rows_tr, cols=("distance",)).sample(200_000, random_state=0)
va_d = add_rates(rows_va, cols=("distance",))
sb, sr = team_shares(va_d[STATE].to_numpy(), va_d[SHAPE].to_numpy(), current.P)
va_d["clo_blue"], va_d["clo_red"] = va_d["alive_blue"] * sb, va_d["alive_red"] * sr
scattered = ((va_d["depth_blue"] > 1.5) | (va_d["width_blue"] > 3)).to_numpy()

closure_rows, regime_rows = [], []
for side, opp in (("blue", "red"), ("red", "blue")):
    geo = [
        "distance",
        f"depth_{side}",
        f"depth_{opp}",
        f"width_{side}",
        f"width_{opp}",
        f"alive_{opp}",
        f"alive_{side}",
    ]
    y = va_d[f"reach_{side}"].to_numpy()
    t_geo = HGB(max_iter=300, random_state=0).fit(tr_d[geo], tr_d[f"reach_{side}"]).predict(va_d[geo])
    more = geo + ["hold", "withdraw", "focus", "d_distance"]
    t_more = HGB(max_iter=300, random_state=0).fit(tr_d[more], tr_d[f"reach_{side}"]).predict(va_d[more])
    form = va_d[f"clo_{side}"].to_numpy()
    closure_rows.append(
        {
            "side": side,
            "formula": r2(y, form),
            "trees, the formula's inputs": r2(y, t_geo),
            "trees, + intent and d′": r2(y, t_more),
            "formula, compact": r2(y[~scattered], form[~scattered]),
            "formula, Blue scattered": r2(y[scattered], form[scattered]),
            "trees, Blue scattered": r2(y[scattered], t_geo[scattered]),
        }
    )
    se_f, se_t = (form - y) ** 2, (t_geo - y) ** 2
    regime_rows.append(
        {
            "side": side,
            "steps with Blue scattered (%)": 100 * scattered.mean(),
            "share of the formula's squared error there (%)": 100 * se_f[scattered].sum() / se_f.sum(),
            "trees' squared error there, same scale (%)": 100 * se_t[scattered].sum() / se_f.sum(),
            "formula bias there (units)": 10 * (form - y)[scattered].mean(),
        }
    )
display(pd.DataFrame(closure_rows).set_index("side").round(3))
pd.DataFrame(regime_rows).set_index("side").round(1)

### Readable closure candidates (per-step fit)

- **F1**: today's form with a saturating enemy-front term, `β·m·c·tanh(depth_opp / c)` instead
  of `β·m·depth_opp` (one more number per side).
- **G2**: built on the root-mean-square unit-to-unit distance `r = √(d² + depths² + widths²)`
  instead of the centroid distance, with the spreads of r from the same moments, and the same
  saturation (three numbers per side).

In [ ]:
RHO = current.P["share_blue"][3]


def cl_inputs(df, side):
    opp = "red" if side == "blue" else "blue"
    cols = ("distance", f"depth_{side}", f"depth_{opp}", f"width_{side}", f"width_{opp}")
    return [df[c].to_numpy() for c in cols] + [10 * df[f"alive_{opp}"].to_numpy()]


def m_of(n):
    return np.sqrt(2 * np.log(np.maximum(n, 1.0)))


def F1(x, lam, beta, g, c):
    d, do, dp, wo, wp, n = x
    return ndtr(
        (RHO - np.sqrt(d**2 + lam * (wo - wp) ** 2) + beta * m_of(n) * c * np.tanh(dp / c))
        / np.sqrt(0.05**2 + (g * do) ** 2 + 1e-4)
    )


def G2(x, beta, g, c):
    d, do, dp, wo, wp, n = x
    rr = d**2 + do**2 + dp**2 + wo**2 + wp**2
    so = np.sqrt((4 * d**2 * do**2 + 2 * do**4 + 2 * wo**4) / (4 * rr) + 1e-6)
    sp = np.sqrt((4 * d**2 * dp**2 + 2 * dp**4 + 2 * wp**4) / (4 * rr) + 1e-6)
    return ndtr((RHO - np.sqrt(rr) + beta * m_of(n) * c * np.tanh(sp / c)) / np.sqrt(0.05**2 + (g * so) ** 2))


cand_fit = rows_tr.sample(120_000, random_state=0)
cand = []
for side in SIDES:
    x, xv = cl_inputs(cand_fit, side), cl_inputs(va_d, side)
    A, Av = cand_fit[f"alive_{side}"].to_numpy(), va_d[f"alive_{side}"].to_numpy()
    y, yv = cand_fit[f"reach_{side}"].to_numpy(), va_d[f"reach_{side}"].to_numpy()
    for name, f, x0 in (
        ("F1 saturating enemy front", F1, [1, 0.6, 1.2, 1.0]),
        ("G2 unit-to-unit distance", G2, [0.6, 1.2, 1.0]),
    ):
        sol = min(
            (
                least_squares(
                    lambda q, A=A, f=f, x=x, y=y: A * f(x, *q) - y, np.array(x0) * k, bounds=(1e-3, 50)
                )
                for k in (1, 0.5, 2)
            ),
            key=lambda s: s.cost,
        )
        pv = Av * f(xv, *sol.x)
        cand.append(
            {
                "side": side,
                "closure": name,
                "R²": r2(yv, pv),
                "R² compact": r2(yv[~scattered], pv[~scattered]),
                "R² scattered": r2(yv[scattered], pv[scattered]),
                "numbers": np.round(sol.x, 2).tolist(),
            }
        )
pd.DataFrame(cand).set_index(["side", "closure"]).round(3)

## 4 Improvement B: the same terms, coefficients fitted on rollout error

`sindyc_06` test 7 showed that fitting on smoothed rates biases the alive rule. Here every
coefficient of `model_derived` is refitted the same way: least squares on the rollout error
(health, alive, distance at 1, 2, 3, 4, 6, 8 s) from 20,000 training start points. **No term is
added or removed.** The decay time of the measured-reach offset (fixed at 1 s today) is fitted
too, per side.

In [ ]:
ALIVE, HEALTH = ["p_b", "p_r", "p_r_focus"], ["kb", "kr", "kr_wd"]
DIST, TAU = ["d_adv", "d_wd", "d_flee", "d_hold_far", "d_hold_contact"], ["tau_b", "tau_r"]
t0 = time.time()
pB, _ = refit(ALIVE + HEALTH + DIST + TAU, pA, Bfit)
print(f"refit in {time.time() - t0:.0f} s")
pd.DataFrame({"model_derived (fitted on rates)": pA, "B (fitted on rollout error)": pB}).loc[
    ALIVE + HEALTH + DIST + TAU
].round(3)

### Tried on top of B, and no better than B (recorded, not re-run here: about 12 minutes)

Each was fitted jointly on rollout error like B, and scored on the validation battles:

| variant | health 4 s | margin R² 4 s | margin R² 8 s |
|---|---|---|---|
| A `model_derived` | 0.055 | 0.073 | −0.656 |
| B refit, same terms | 0.052 | 0.281 | −0.440 |
| A + formation equations as fitted (on rates) | 0.056 | 0.056 | −0.793 |
| A + formation equations refitted multi-step | 0.056 | 0.009 | −0.753 |
| B + formation equations (Blue width held) | 0.051 | 0.273 | −0.562 |
| … + the closure's λ, β, g refitted on rollout error | 0.048 | 0.286 | −0.546 |
| … + saturating enemy front (F1) | 0.048 | 0.293 | −0.535 |

The formation equations themselves do predict formation better than holding it (Red depth error
at 4 s 0.20 → 0.11, Red width 0.31 → 0.24 after a multi-step refit; Blue width no better), but
with the formula closure that does not carry through to health or margin.

## 5 Improvement S: the share in reach as a state

In `sindyc_06` (tests 2–3) the share showed no memory *given the true current state*. But in a
rollout the current distance and formation are the model's own, and wrong. Bound first: trees
predict each side's share path directly from the start state (as in section 1), and
`model_derived`'s own health and alive equations run on that share.

In [ ]:
def share_path(Bx):
    Y, R = carry(Bx["Y"]), carry(Bx["Yreach"])
    return np.divide(R, Y[:, :, 2:4], out=np.zeros_like(R), where=Y[:, :, 2:4] > 0)


s_tr = share_path(Bt)
pred_share = np.zeros_like(share_true)
pred_share[:, 0] = share_true[:, 0]
knots = list(range(4, 65, 4))
for n in knots:
    ok = ~np.isnan(Bt["Y"][:, n, 0])
    ft = np.column_stack([start_features(Bt, n), s_tr[:, 0]])
    fv = np.column_stack([start_features(Bv, n), share_true[:, 0]])
    for j in (0, 1):
        pred_share[:, n, j] = np.clip(
            HGB(max_iter=200, random_state=0).fit(ft[ok], s_tr[ok, n, j]).predict(fv), 0, 1
        )
prev = 0
for n in knots:
    for k in range(prev + 1, n):
        w = (k - prev) / (n - prev)
        pred_share[:, k] = (1 - w) * pred_share[:, prev] + w * pred_share[:, n]
    prev = n
bound = {
    "A model_derived (closure on its own distance)": metrics(simulate(pA, Bv), Bv),
    "A with the share predicted from the start state (trees)": metrics(
        simulate(pA, Bv, truth={"share": pred_share}), Bv
    ),
    "A with the true share": metrics(simulate(pA, Bv, truth={"share": share_true}), Bv),
}
show(bound)

### A readable equation for the share

The share s of each side becomes a state, started at the measured share:

    s' = sa·(closure)' + sk·(closure − s)                    follow the geometry / relax to it
         + C·(1 − s)·(sf_adv·adv + sf_hold·hold + sf_wd·wd)   in contact, units still outside move in
         − so_wd·wd·s                                         withdrawing empties the reach
         + sd·(1 − s)·(ln A)'                                 deaths change the share

C is the chance that any Blue unit is in reach (already used by the distance equation). With
sa = 1, sk = 1/τ and the rest 0 this **is** the baseline's decaying offset, so the baseline is
nested. Seven candidates per side, fitted on rollout error (now including units in reach) with
B's other numbers held; then pruned: a term is dropped if removing it raises the loss by less
than 1%, and the rest refitted.

In [ ]:
SH = [f"{k}_{t}" for t in "br" for k in ("sa", "sk", "sf_adv", "sf_hold", "sf_wd", "so_wd", "sd")]


def as_share_state(p):
    q = dict(p, share_state=True)
    for t in "br":
        q.update(
            {
                f"sa_{t}": 1.0,
                f"sk_{t}": 1 / p[f"tau_{t}"],
                f"sf_adv_{t}": 0.0,
                f"sf_hold_{t}": 0.0,
                f"sf_wd_{t}": 0.0,
                f"so_wd_{t}": 0.0,
                f"sd_{t}": 0.0,
            }
        )
    return q


print(
    "nesting check, max |difference| to B:",
    np.nanmax(np.abs(simulate(as_share_state(pB), Bv) - simulate(pB, Bv))).round(3),
    "(small: the two discretisations differ slightly)",
)
t0 = time.time()
pS14, _ = refit(SH, as_share_state(pB), Bfit, reach_w=1.0)
loss = lambda p: float((residuals([], [], p, Bfit, reach_w=1.0) ** 2).sum())  # noqa: E731
base_loss = loss(pS14)
importance = pd.Series({k: loss(dict(pS14, **{k: 0.0})) / base_loss - 1 for k in SH}).sort_values()
keep = [k for k in SH if importance[k] >= 0.01]
pS, _ = refit(keep, dict(pS14, **{k: 0.0 for k in SH if k not in keep}), Bfit, reach_w=1.0)
print(f"fitted and pruned in {time.time() - t0:.0f} s; kept {len(keep)} of {len(SH)} terms")
pd.DataFrame(
    {
        "loss increase if removed": importance,
        "all 14 terms": {k: pS14[k] for k in SH},
        "S (pruned, refitted)": {k: pS[k] for k in SH},
    }
).loc[SH].round(3)

## 6 Verification

### 6a Validation and the untouched test set, with paired bootstrap intervals

Intervals resample battles (the test set: the 600 seeds, so the six branches of a battle stay
together). "diff" is new − old on the same resample; an interval that excludes 0 is a real
difference.

In [ ]:
models = {
    "A model_derived": pA,
    "B refit, same terms": pB,
    "S share as a state": pS,
    "S with all 14 terms": pS14,
}
pickle.dump(models, open(PROCESSED / "sindyc_07_models.pkl", "wb"))


def reach_error(p, Bx):
    _, R = simulate(p, Bx, return_reach=True)
    return {
        f"reach {h}s (units)": 10
        * np.nanmean(np.abs(R[:, round(h / DT) - 1] - Bx["Yreach"][:, round(h / DT) - 1]))
        for h in H
    }


rolls, verify = {}, []
for name in ("val", "test"):
    Bx = B[name]
    rolls[name] = {k: simulate(p, Bx) for k, p in models.items()}
    assert all(np.isfinite(Y).all() for Y in rolls[name].values())
    T = pd.DataFrame(
        {
            k: pd.concat([metrics(Y, Bx), pd.Series(reach_error(models[k], Bx))])
            for k, Y in rolls[name].items()
        }
    ).T
    print(f"=== {name}")
    display(T[[c for c in T.columns if "health" in c or "margin" in c or "reach" in c]].round(3))
    for h in (2, 4, 8):
        for a, b in (
            ("A model_derived", "B refit, same terms"),
            ("A model_derived", "S share as a state"),
            ("B refit, same terms", "S share as a state"),
        ):
            ci = paired_bootstrap(rolls[name][a], rolls[name][b], Bx, h=h, n_boot=1000)
            verify.append(
                {
                    "set": name,
                    "horizon (s)": h,
                    "comparison": f"{b.split()[0]} vs {a.split()[0]}",
                    "health diff, 95%": f"{ci['health b-a'][0]:+.4f} … {ci['health b-a'][1]:+.4f}",
                    "margin R² old, 95%": f"{ci['margin a'][0]:.2f} … {ci['margin a'][1]:.2f}",
                    "margin R² new, 95%": f"{ci['margin b'][0]:.2f} … {ci['margin b'][1]:.2f}",
                    "margin R² diff, 95%": f"{ci['margin b-a'][0]:+.2f} … {ci['margin b-a'][1]:+.2f}",
                }
            )
pd.DataFrame(verify).set_index(["set", "horizon (s)", "comparison"])

### 6b Overfitting, stability, and the intent ranking

- **Overfitting:** the same metrics on training start points that were not used for fitting.
- **Stability:** S's kept terms refitted on a different training subsample.
- **Intent ranking** (as in `sindyc_03`): for each counterfactual state, the model's predicted
  final margin under each of the six intents (held 4 s, then advance/nearest) against the real
  one. Not a goal of this notebook (RQ5 is out of scope); included as an independent check.

In [ ]:
Bother = subsample(B["train"], 20000, seed=7)
over = {
    k: {
        "train: margin R² 4 s": metrics(simulate(p, Bother), Bother)["margin R2 4s"],
        "val: margin R² 4 s": metrics(rolls["val"][k], Bv)["margin R2 4s"],
        "train: health 4 s": metrics(simulate(p, Bother), Bother)["health 4s"],
        "val: health 4 s": metrics(rolls["val"][k], Bv)["health 4s"],
    }
    for k, p in models.items()
}
display(pd.DataFrame(over).T.round(3))

pS2, _ = refit(keep, pS, subsample(B["train"], 20000, seed=3), reach_w=1.0)
display(
    pd.DataFrame(
        {"S (subsample 0)": {k: pS[k] for k in keep}, "S (subsample 3)": {k: pS2[k] for k in keep}}
    ).round(3)
)
print(
    "subsample-3 model on validation:",
    metrics(simulate(pS2, Bv), Bv)[["health 4s", "margin R2 4s", "margin R2 8s"]].round(3).to_dict(),
)

LABELS = [f"{m}/{t}" for m, t in INTENTS]
tables = {i: pd.read_parquet(PROCESSED / f"cf_{i[0]}_{i[1]}.parquet") for i in INTENTS}
recs, starts = [], []
for ep, g in tables[INTENTS[0]].groupby(level=0):
    k = int(round(g["branch_at"].iloc[0] / DT))
    if k >= len(g) - 8 or g["alive_blue"].iloc[k] <= 0 or g["alive_red"].iloc[k] <= 0:
        continue
    rec = {}
    for label, i in zip(LABELS, INTENTS, strict=True):
        h = tables[i].loc[ep]
        rec[f"final {label}"] = h["health_blue"].iloc[-1] - h["health_red"].iloc[-1]
        rec[f"win {label}"] = bool(h["battle_won"].iloc[0])
    recs.append(rec)
    starts.append(g.iloc[k][STATE + SHAPE + REACH_COLS])
cf, st = pd.DataFrame(recs), pd.DataFrame(starts).reset_index(drop=True)
Tm, W = cf[[f"final {x}" for x in LABELS]].to_numpy(), cf[[f"win {x}" for x in LABELS]].to_numpy()
idx = np.arange(len(cf))
ranking = {
    "always the best fixed intent": {"win rate": W.mean(0).max()},
    "hindsight best": {"win rate": W.max(1).mean()},
}
for k, p in models.items():
    pm = []
    for i in INTENTS:
        U = np.repeat(control_vector("advance", "nearest")[None], 128, 0)
        U[:32] = control_vector(*i)
        Bc = {
            "X0": st[STATE].to_numpy(float),
            "shape0": st[SHAPE].to_numpy(float),
            "reach0": st[REACH_COLS].to_numpy(float),
            "U": np.repeat(U[None], len(st), 0),
        }
        Y = simulate(p, Bc)
        pm.append(Y[:, -1, 0] - Y[:, -1, 1])
    pm = np.column_stack(pm)
    ch = pm.argmax(1)
    ranking[k] = {
        "win rate": W[idx, ch].mean(),
        "rank correlation": np.nanmean([spearmanr(pm[n], Tm[n]).statistic for n in idx]),
        "picks withdraw": np.mean([LABELS[c].startswith("withdraw") for c in ch]),
        "regret": (Tm.max(1) - Tm[idx, ch]).mean(),
    }
print(f"{len(cf)} counterfactual states")
pd.DataFrame(ranking).T.round(3)

## 7 Summary of the numbers (test set, 4 s)

In [ ]:
summary = pd.DataFrame(
    {
        k: {
            "health error 4 s": metrics(rolls["test"][k], B["test"])["health 4s"],
            "margin R² 2 s": metrics(rolls["test"][k], B["test"])["margin R2 2s"],
            "margin R² 4 s": metrics(rolls["test"][k], B["test"])["margin R2 4s"],
            "margin R² 8 s": metrics(rolls["test"][k], B["test"])["margin R2 8s"],
        }
        for k in models
    }
).T
summary.loc["ceiling (trees from the start state)"] = [
    ceiling.loc[("test", 4), "health: trees"],
    ceiling.loc[("test", 2), "margin R²: trees"],
    ceiling.loc[("test", 4), "margin R²: trees"],
    ceiling.loc[("test", 8), "margin R²: trees"],
]
summary.round(3)

## 8 Transfer to stalkers without refitting (RQ4)

`model_derived` transfers to stalkers by scaling with three game constants (`switch_library.scaled`):
distances with reach, firepower with 1 / (time to kill), closing speeds with walking speed.
Do B and S transfer the same way? B's and S's coefficients were fitted on marine rollouts, so they
could have absorbed marine-specific errors.

S has rates per second that the existing rule doesn't cover. **Rule fixed before looking at any
stalker result:** they scale with the movement time (reach / walking speed), because units
entering and leaving reach is movement. Two alternatives are shown as a sensitivity check: no
scaling, and scaling with the time to kill.

References: the same structures with coefficients refitted on the stalker training battles
(starting from the scaled values; closure and gate shapes kept scaled), and the tree ceiling
trained on stalker battles. Stalker battles last about 30 s, so the horizon goes to 16 s.

In [ ]:
from sindy_wm.models.switch_library import UNITS, scaled  # noqa: E402

Bs = lab.load_batch("val", "exp07_stalkers", horizon=128)
Bs_train = lab.load_batch("train", "exp07_stalkers", horizon=128)
print(
    f"stalker validation: {len(Bs['X0'])} start points from {len(np.unique(Bs['group']))} battles; "
    f"training: {len(Bs_train['X0'])}"
)
display(pd.DataFrame(UNITS).T.round(2))
lib_scaled = scaled(current, "marine", "stalker").simulate(
    Bs["X0"], Bs["shape0"], Bs["U"], reach0=Bs["reach0"]
)
print(
    "lean scaled simulator vs switch_library.scaled, max |difference|:",
    np.nanmax(np.abs(simulate(scale_params(pA, rates="none"), Bs) - lib_scaled)),
)

HS = (1, 2, 4, 8, 16)
stalker, Ys = {}, {}


def record(label, p):
    Ys[label] = simulate(p, Bs)
    stalker[label] = metrics(Ys[label], Bs, horizons=HS)


record("A marine model, as is", pA)
record("A scaled", scale_params(pA, rates="none"))
record("B scaled", scale_params(pB))
record("S scaled", scale_params(pS))
record("S scaled, rates not scaled", scale_params(pS, rates="none"))
record("S scaled, rates by time to kill", scale_params(pS, rates="combat"))
Bs_fit = subsample(Bs_train, 20000)
CORE = ALIVE + HEALTH + DIST
rB, _ = refit(CORE + TAU, scale_params(pB), Bs_fit)
rS, _ = refit(CORE + keep, scale_params(pS), Bs_fit, reach_w=1.0)
record("B refitted on stalkers", rB)
record("S refitted on stalkers", rS)
for h in HS:
    n = round(h / DT)
    okt, ok = ~np.isnan(Bs_train["Y"][:, n, 0]), ~np.isnan(Bs["Y"][:, n, 0])
    Xt, Xv = start_features(Bs_train, n)[okt], start_features(Bs, n)[ok]
    pr = [
        HGB(max_iter=300, random_state=0)
        .fit(Xt, Bs_train["Y"][okt, n, j] - Bs_train["X0"][okt, j])
        .predict(Xv)
        + Bs["X0"][ok, j]
        for j in (0, 1)
    ]
    real = Bs["Y"][ok, n, 0] - Bs["Y"][ok, n, 1] - (Bs["X0"][ok, 0] - Bs["X0"][ok, 1])
    pm = pr[0] - pr[1] - (Bs["X0"][ok, 0] - Bs["X0"][ok, 1])
    stalker.setdefault("ceiling (trees, stalker-trained)", {})[f"health {h}s"] = np.mean(
        [np.abs(pr[j] - Bs["Y"][ok, n, j]).mean() for j in (0, 1)]
    )
    stalker["ceiling (trees, stalker-trained)"][f"margin R2 {h}s"] = 1 - np.sum((pm - real) ** 2) / np.sum(
        (real - real.mean()) ** 2
    )
T = pd.DataFrame(stalker).T
display(T[[c for c in T.columns if "health" in c]].round(3))
display(T[[c for c in T.columns if "margin" in c]].round(3))

transfer_ci = []
for h in (4, 8, 16):
    for a, b in (("A scaled", "B scaled"), ("A scaled", "S scaled"), ("B scaled", "S scaled")):
        ci = paired_bootstrap(Ys[a], Ys[b], Bs, h=h, n_boot=1000)
        transfer_ci.append(
            {
                "horizon (s)": h,
                "comparison": f"{b} vs {a}",
                "health diff, 95%": f"{ci['health b-a'][0]:+.4f} … {ci['health b-a'][1]:+.4f}",
                "margin R² diff, 95%": f"{ci['margin b-a'][0]:+.2f} … {ci['margin b-a'][1]:+.2f}",
            }
        )
display(pd.DataFrame(transfer_ci).set_index(["horizon (s)", "comparison"]))
pd.DataFrame(
    {
        "S scaled from marines": {k: scale_params(pS)[k] for k in CORE + keep},
        "S refitted on stalkers": {k: rS[k] for k in CORE + keep},
    }
).round(3)

## 9 Weak-form SINDy: is B's gain bias correction or compensation?

B's coefficients moved for two possible reasons: (1) fitting on smoothed rates biases them
(attenuation, shown for the alive rule in `sindyc_06`), and fitting on rollouts removes that;
(2) rollout fitting lets coefficients compensate for errors elsewhere (the closure). Weak-form
SINDy (WSINDy; Messenger & Bortz 2021, after Schaeffer & McCalla 2017) separates the two: it fits
the same library with the same STLSQ thresholds, but on integrals against smooth test functions
instead of derivatives, so it removes (1) without allowing (2).

Reproduction first: `model_derived` was fitted with measured reach in the health library and a
health threshold of 0.15 (re-fitting with those settings gives its coefficients exactly).
Test functions: φ = (1 − s²)⁴ on windows of radius 0.5, 1 or 2 s, every 2 steps, inside the part
of each training battle where both sides are alive; for the alive rule only windows with a
constant intent. Radius chosen on validation; then the test set and stalkers once.
Code: `notebooks/weak_sindy.py`.

In [ ]:
from weak_sindy import fit_weak, weak_rows  # noqa: E402


def coef_table(m):
    d = {}
    for eq in ("health_blue", "health_red", "distance"):
        d |= {f"{eq}: {n}": c for n, c in zip(m.names, m.coef[eq], strict=True) if c}
    for side in SIDES:
        d |= {f"alive {side}: {k}": v for k, v in m.p[side].items() if v}
    return pd.Series(d)


weak_models, weak_val = {}, {}
coefs = {"rates (model_derived)": coef_table(current)}
for radius in (4, 8, 16):
    Wrows = weak_rows(rows_tr, current.P, current.names, radius=radius)
    m = fit_weak(current, Wrows)
    label = f"weak, radius {radius * DT:g} s"
    weak_models[label], coefs[label] = m, coef_table(m)
    weak_val[label] = metrics(m.simulate(Bv["X0"], Bv["shape0"], Bv["U"], reach0=Bv["reach0"]), Bv)
coefs["B (rollout)"] = pd.Series(
    {
        "health_blue: Rr": pB["kb"],
        "health_red: Rb": pB["kr"],
        "health_red: Rb*wd": pB["kr_wd"],
        "alive blue: 1": pB["p_b"],
        "alive red: 1": pB["p_r"],
        "alive red: focus": pB["p_r_focus"],
        "distance: close*adv": pB["d_adv"],
        "distance: close*wd": pB["d_wd"],
        "distance: flee*wd": pB["d_flee"],
        "distance: close*hold*far": pB["d_hold_far"],
        "distance: close*hold*contact": pB["d_hold_contact"],
    }
)
display(pd.DataFrame(coefs).round(3))
weak_val["A model_derived"], weak_val["B refit, same terms"] = (
    metrics(rolls["val"]["A model_derived"], Bv),
    metrics(rolls["val"]["B refit, same terms"], Bv),
)
show(weak_val)

The largest window is best on validation by a small margin; it goes to the test set and to
stalkers (scaled as before), against A with paired intervals.

In [ ]:
Wbest = weak_models["weak, radius 2 s"]
weak_check, weak_ci = {}, []
for name in ("val", "test"):
    Bx = B[name]
    YW = Wbest.simulate(Bx["X0"], Bx["shape0"], Bx["U"], reach0=Bx["reach0"])
    weak_check[(name, "A")], weak_check[(name, "W weak form")], weak_check[(name, "B")] = (
        metrics(rolls[name]["A model_derived"], Bx),
        metrics(YW, Bx),
        metrics(rolls[name]["B refit, same terms"], Bx),
    )
    for h in (2, 4, 8):
        ci = paired_bootstrap(rolls[name]["A model_derived"], YW, Bx, h=h, n_boot=1000)
        weak_ci.append(
            {
                "set": name,
                "horizon (s)": h,
                "health diff W − A, 95%": f"{ci['health b-a'][0]:+.4f} … {ci['health b-a'][1]:+.4f}",
                "margin R² diff W − A, 95%": f"{ci['margin b-a'][0]:+.3f} … {ci['margin b-a'][1]:+.3f}",
            }
        )
YWs = scaled(Wbest, "marine", "stalker").simulate(Bs["X0"], Bs["shape0"], Bs["U"], reach0=Bs["reach0"])
weak_check[("stalkers", "A scaled")], weak_check[("stalkers", "W scaled")] = (
    metrics(Ys["A scaled"], Bs, horizons=(1, 2, 4, 8)),
    metrics(YWs, Bs, horizons=(1, 2, 4, 8)),
)
for h in (4, 8):
    ci = paired_bootstrap(Ys["A scaled"], YWs, Bs, h=h, n_boot=1000)
    weak_ci.append(
        {
            "set": "stalkers",
            "horizon (s)": h,
            "health diff W − A, 95%": f"{ci['health b-a'][0]:+.4f} … {ci['health b-a'][1]:+.4f}",
            "margin R² diff W − A, 95%": f"{ci['margin b-a'][0]:+.3f} … {ci['margin b-a'][1]:+.3f}",
        }
    )
display(show(weak_check, cols=("health", "margin", "alive")))
pd.DataFrame(weak_ci).set_index(["set", "horizon (s)"])

### 9b The switch shapes in weak form too

The 24 numbers of the movement switches were fitted with `curve_fit` on the smoothed distance
rate, so they could carry the same rate bias. Here they are refitted in weak form (same
functions, current values as the start, windows where the movement is constant), at radius 1 and
2 s; then the weak SINDy fit of section 9 is repeated on the new switches ("WG", fully weak
form). Window radius chosen on validation; then test and stalkers.

In [ ]:
import copy  # noqa: E402

from weak_sindy import _gate_windows, fit_gates_weak  # noqa: E402

from sindy_wm.models.intent_model import _close_flee_gate, _close_gate  # noqa: E402

wg_models, wg_val = {}, {}
for radius in (8, 16):
    P2 = fit_gates_weak(rows_tr, current.P, radius=radius)
    base = copy.deepcopy(current)
    base.P = P2
    m = fit_weak(base, weak_rows(rows_tr, P2, current.names, radius=16))
    label = f"WG, switch windows {radius * DT:g} s"
    wg_models[label] = m
    wg_val[label] = metrics(m.simulate(Bv["X0"], Bv["shape0"], Bv["U"], reach0=Bv["reach0"]), Bv)
wg_val["W weak form"], wg_val["A model_derived"] = weak_val["weak, radius 2 s"], weak_val["A model_derived"]
display(show(wg_val, cols=("health", "margin", "distance")))

WG = wg_models["WG, switch windows 2 s"]
sw = []
for mv, f in (("advance", _close_gate), ("hold", _close_gate), ("withdraw", _close_flee_gate)):
    st, Phi, D = _gate_windows(rows_va, mv, 16)
    Xg = tuple(st[c].to_numpy() for c in ["distance"] + SHAPE)
    bw = D @ st["distance"].to_numpy()
    sw.append(
        {
            "movement": mv,
            "switch (old)": np.round(current.P["gates"][mv], 2).tolist(),
            "switch (weak)": np.round(WG.P["gates"][mv], 2).tolist(),
            "weak-form R² on validation windows, old": r2(bw, Phi @ f(Xg, *current.P["gates"][mv])),
            "new": r2(bw, Phi @ f(Xg, *WG.P["gates"][mv])),
        }
    )
display(pd.DataFrame(sw).set_index("movement").round(3))
print(
    "distance equation, WG:",
    {n: round(c, 3) for n, c in zip(WG.names, WG.coef["distance"], strict=True) if c},
)

wg_check, wg_ci = {}, []
for name, Bx, mods in (
    ("test", B["test"], {"A": current, "W": Wbest, "WG": WG}),
    (
        "stalkers",
        Bs,
        {k: scaled(m, "marine", "stalker") for k, m in {"A": current, "W": Wbest, "WG": WG}.items()},
    ),
):
    Yx = {k: m.simulate(Bx["X0"], Bx["shape0"], Bx["U"], reach0=Bx["reach0"]) for k, m in mods.items()}
    for k, v in Yx.items():
        wg_check[(name, k)] = metrics(v, Bx, horizons=(2, 4, 8))
    for h in (4, 8):
        for a, b2 in (("A", "WG"), ("W", "WG")):
            ci = paired_bootstrap(Yx[a], Yx[b2], Bx, h=h, n_boot=1000)
            wg_ci.append(
                {
                    "set": name,
                    "horizon (s)": h,
                    "comparison": f"{b2} vs {a}",
                    "health diff, 95%": f"{ci['health b-a'][0]:+.4f} … {ci['health b-a'][1]:+.4f}",
                    "margin R² diff, 95%": f"{ci['margin b-a'][0]:+.3f} … {ci['margin b-a'][1]:+.3f}",
                }
            )
display(show(wg_check, cols=("health", "margin", "distance")))
pd.DataFrame(wg_ci).set_index(["set", "horizon (s)", "comparison"])

For reference, B's gain by stage (section 4's refit run one group at a time, validation, recorded
from the exploratory run): alive exponents only, margin R² at 4 s 0.073 → 0.080; + health
coefficients 0.215; + distance 0.220; + offset decay times 0.281.

## Questions (for you)

1. Section 1–2: the trees reach a margin R² of about 0.63–0.70 at 4 s from the same state, and
   `model_derived` with the true share reaches the same. What does that say about RQ2 (is the
   team-level state enough?) compared with what the status note says today?
2. Section 3: the closure has the inputs it needs, but its form fails when Blue is scattered.
   Why does a centroid distance stop describing a team that has withdrawn into an arc?
3. Section 4: B changes no term, only how the numbers are fitted. The health coefficients moved
   *down* (0.177 → 0.156) while the alive exponents moved up. Does that fit the attenuation story
   from `sindyc_06`, or is something else going on? (Hint: what does a rollout-fitted coefficient
   compensate for that a rate-fitted one does not?)
4. Section 5: `sindyc_06` said R needs no memory; here the share as a state helps. Are those in
   conflict? Write down in your own words why both can be true.
5. Read S's kept terms and signs. Which ones can you explain physically, and which not? (Red's
   `sd` is large and negative; Red's fan-out terms are negative.)
6. S closes roughly half the distance from A to the ceiling on the margin, but the health error
   moves little (0.058 → 0.054, ceiling 0.034). What would you look at next, and is it worth it
   within the six-week plan?
7. Section 8: B and S scaled to stalkers do better than `model_derived` scaled. Compare S's scaled
   coefficients with the ones refitted on stalkers: which agree, and which don't (look at the
   distance and share terms)? What does that say about which parts of the model are "physics that
   scales" and which are specific to marines?
8. Section 9: weak-form SINDy moves the alive exponents towards B's values but keeps the health
   coefficients near the rate fit, and the margin gets slightly *worse*. Where, then, does B's
   gain come from? And why can removing a bias make the predictions worse?
9. Section 9b: the weak-form switches fit the distance windows much better, but the rollouts of
   distance barely change and some numbers grow large (closing speeds, switch widths near 0).
   What does a better fit of d′ that does not give a better distance forecast tell you?